# Detail compression by span search: change with source age

For each summary detail, the search finds the best consecutive run of source details. The size of that run is the
compression of that detail. This notebook asks how that compression changes with **source age**, for the three
summaries of each story (younger, actual, older), under three similarity measures:

- **natural language**: mpnet text embeddings
- **contextual**: GPT-OSS detail vectors pooled from a pass over the whole text
- **isolated**: GPT-OSS detail vectors from each detail encoded alone

Inputs are written by `hpc/analysis/submit_span_search.sh`. See `docs/detail_compression_span_search.md`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path(os.environ.get("RESULTS_DIR", "results"))
NL_DIR = RESULTS / "analysis/natural_language/detail_compression/span_search"
REP_DIR = RESULTS / "analysis/representation/detail_compression/span_search"
SOURCES = {
    "natural language": NL_DIR,
    "contextual": REP_DIR / "contextual_mean",
    "isolated": REP_DIR / "isolated_mean",
}
PENALTY = 0.1      # length penalty used for the main figures (the search was run at several)
LAYER = 12         # representation layer for the main figures (nearest stored layer is used)
AGE_BIN = 1.0      # width of the source-age bins, in years (coverage table only)
BANDWIDTH = 0.75   # width of the Gaussian kernel that smooths the curves over source age, in years
COMMON_STORIES_ONLY = True   # compare methods on the stories every method covers (vectors exist for fewer stories than the text search)
CONDITIONS = ["younger", "actual", "older"]
COLORS = {"younger": "#2a78d6", "actual": "#eb6834", "older": "#1baf7a"}   # fixed order, never cycled
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e5e1"
import sys
for _here in [Path(os.environ.get("STORY_FEATURES_ROOT", ".")).resolve(), Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_here / "src/shared/retention_shift.py").exists():
        sys.path.insert(0, str(_here / "src"))
        break
from shared.retention_shift import smooth_with_band
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": MUTED,
    "axes.labelcolor": "#52514e", "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 110, "font.size": 10,
})

In [ ]:
def nearest(values, target):
    values = np.asarray(sorted(set(values)))
    return values[np.argmin(np.abs(values - target))]

summaries, details, available = [], [], {}
for method, folder in SOURCES.items():
    if not (folder / "summaries.csv").exists():
        print(f"missing: {folder}")
        continue
    s, d = pd.read_csv(folder / "summaries.csv"), pd.read_csv(folder / "details.csv")
    for frame in (s, d):
        frame["method"] = method
    available[method] = (s, d)
    summaries.append(s); details.append(d)
summaries, details = pd.concat(summaries, ignore_index=True), pd.concat(details, ignore_index=True)
if "layer" not in summaries:
    summaries["layer"], details["layer"] = np.nan, np.nan

penalty = nearest(summaries.length_penalty, PENALTY)
layer_values = summaries.layer.dropna().unique()
layer = nearest(layer_values, LAYER) if len(layer_values) else np.nan
print(f"length penalty {penalty}, representation layer {layer}")

def main_view(frame):
    keep = frame.length_penalty.eq(penalty)
    rep = frame.method.ne("natural language")
    return frame[keep & (~rep | frame.layer.eq(layer))].copy()

S, D = main_view(summaries), main_view(details)
if COMMON_STORIES_ONLY and S.method.nunique() > 1:
    per_method = {m: set(g.story_key) for m, g in S.groupby("method")}
    print({m: len(s) for m, s in per_method.items()}, "stories per method")
    common = set.intersection(*per_method.values())
    S, D = S[S.story_key.isin(common)].copy(), D[D.story_key.isin(common)].copy()
    print(f"{len(common)} stories shared by all methods")
S["story_age"] = S.source_age
print(S.groupby(["method", "condition"]).story_key.nunique().unstack())

## Coverage by source age
Stories per age bin. Bins with only a few stories give wide intervals below.

In [ ]:
S["age_bin"] = np.floor(S.source_age / AGE_BIN) * AGE_BIN
coverage = S[S.method.eq("natural language") | S.method.eq("contextual")].drop_duplicates(["method", "story_key"])
coverage.groupby(["method", "age_bin"]).story_key.nunique().unstack(0)

In [ ]:
def binned(frame, y, n_boot=400, seed=0):
    """Mean of y per condition and age bin, with a story-level bootstrap 95% interval."""
    rng = np.random.default_rng(seed)
    rows = []
    for (condition, age_bin), group in frame.groupby(["condition", "age_bin"]):
        per_story = group.groupby("story_key")[y].mean().to_numpy()
        if len(per_story) == 0:
            continue
        boots = rng.choice(per_story, size=(n_boot, len(per_story))).mean(axis=1)
        rows.append({"condition": condition, "age_bin": age_bin, "mean": per_story.mean(),
                     "lo": np.percentile(boots, 2.5), "hi": np.percentile(boots, 97.5), "n": len(per_story)})
    return pd.DataFrame(rows)


def age_panels(frame, y, ylabel, title, sharey=True, zero_line=False):
    """Compression measure against source age: one value per story, kernel-smoothed, with a story-resampling 95% band."""
    methods = [m for m in SOURCES if m in set(frame.method)]
    fig, axes = plt.subplots(1, len(methods), figsize=(5.2 * len(methods), 4.2), sharey=sharey, squeeze=False)
    for ax, method in zip(axes[0], methods):
        data = frame[frame.method.eq(method)]
        ages = data.drop_duplicates("story_key").source_age
        lo, hi = np.quantile(ages, [0.03, 0.97])
        grid = np.round(np.arange(np.ceil(lo * 2) / 2, hi + 1e-9, 0.25), 2)
        for condition in CONDITIONS:
            sub = data[data.condition.eq(condition)]
            if sub.empty:
                continue
            table = sub.groupby("story_key").agg(source_age=("source_age", "first"), value=(y, "mean")).reset_index()
            table["weight"] = 1.0
            ax.scatter(table.source_age, table.value, s=7, alpha=0.15, color=COLORS[condition], linewidths=0)
            curve = smooth_with_band(table, "value", grid, BANDWIDTH, n_boot=300)
            ax.fill_between(curve.age, curve.lo, curve.hi, color=COLORS[condition], alpha=0.15, linewidth=0)
            ax.plot(curve.age, curve.estimate, color=COLORS[condition], lw=2.2, label=condition)
        if zero_line:
            ax.axhline(0, color=MUTED, lw=1)
        ax.plot(ages, np.full(len(ages), ax.get_ylim()[0]), "|", color="#555555", alpha=0.25, markersize=8)
        ax.set_xlim(grid.min(), grid.max())
        ax.set_title(method, loc="left", fontsize=10)
        ax.set_xlabel("Source age (years)")
    axes[0][0].set_ylabel(ylabel)
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", ncol=3, frameon=False, title="summary")
    fig.suptitle(title, x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    return fig

## 1. Compression in detail counts (primary)
Mean number of source details matched by one summary detail, per summary. Higher means each summary detail stands for more of the story.

In [ ]:
age_panels(S, "mean_span_len", "Source details per summary detail", "Mean span length by source age");

## 2. Normalised per story
`span_share` is the matched span as a share of the story's details. `relative_compression` divides out the summary's own overall compression, so 1.0 means an average amount for that summary.

In [ ]:
age_panels(S, "mean_span_share", "Share of story's details", "Matched span as a share of the story");
age_panels(S, "mean_relative_compression", "Relative compression (1 = this summary's average)", "Relative compression", zero_line=False);

## 3. Compression in words and in the summary as a whole
Log2 word ratio: 0 means the matched span has the same number of words as the summary detail; +1 means twice as many.

In [ ]:
age_panels(S, "mean_log2_word_ratio", "log2(span words / summary-detail words)", "Word ratio by source age", zero_line=True);
age_panels(S, "detail_compression", "Source details / summary details", "Overall detail compression by source age");

## 4. Does the match beat chance?
Match score minus the score from the same search against a different story's source (in score-spread units). Near zero means the span is not better than a random story's span, and the ratios above should not be interpreted.

In [ ]:
S["score_over_null"] = S.mean_score - S.mean_null_score
age_panels(S, "score_over_null", "Match score - null score", "Match quality above chance", sharey=False, zero_line=True);

## 5. Younger vs actual vs older within the same story
Paired difference in mean span length. Positive `older - actual` means older-target summaries compress each detail more.

In [ ]:
wide = S.pivot_table(index=["method", "story_key", "source_age"], columns="condition", values="mean_span_len").reset_index()
wide["older - actual"] = wide.get("older") - wide.get("actual")
wide["actual - younger"] = wide.get("actual") - wide.get("younger")
methods = [m for m in SOURCES if m in set(wide.method)]
fig, axes = plt.subplots(1, len(methods), figsize=(4.2 * len(methods), 3.6), sharey=True, squeeze=False)
for ax, method in zip(axes[0], methods):
    data = wide[wide.method.eq(method)]
    for column, color in [("actual - younger", COLORS["younger"]), ("older - actual", COLORS["older"])]:
        sub = data.dropna(subset=[column])
        ax.scatter(sub.source_age, sub[column], s=9, alpha=0.25, color=color, linewidths=0)
        sub = sub.assign(age_bin=np.floor(sub.source_age / AGE_BIN) * AGE_BIN)
        line = sub.groupby("age_bin")[column].mean()
        ax.plot(line.index + AGE_BIN / 2, line.values, color=color, lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=column)
    ax.axhline(0, color=MUTED, lw=1)
    ax.set_title(method, loc="left", fontsize=10); ax.set_xlabel("Source age (years)")
axes[0][0].set_ylabel("Difference in mean span length")
axes[0][0].legend(frameon=False)
fig.tight_layout();

## 6. How steep is the age trend?
Slope of mean span length on source age, per condition, with the story as the unit. A slope of 0.1 means one extra source detail per summary detail for every 10 years of source age.

In [ ]:
from scipy.stats import linregress, t as student_t

def age_slope(group, y="mean_span_len"):
    """OLS slope of y on source age (one row per story), with a 95% interval."""
    fit = linregress(group.source_age, group[y])
    half = student_t.ppf(0.975, len(group) - 2) * fit.stderr
    return fit.slope, fit.slope - half, fit.slope + half, fit.pvalue

rows = []
for (method, condition), group in S.groupby(["method", "condition"]):
    group = group.dropna(subset=["mean_span_len", "source_age"])
    if group.story_key.nunique() < 4:
        continue
    slope, low, high, p = age_slope(group)
    rows.append({"method": method, "condition": condition, "stories": group.story_key.nunique(),
                 "slope per year": slope, "ci low": low, "ci high": high, "p": p})
slopes = pd.DataFrame(rows)
slopes.round(3)

In [ ]:
if len(slopes):
    fig, ax = plt.subplots(figsize=(6.5, 0.7 * len(slopes) + 1.4))
    labels = slopes.method + " / " + slopes.condition
    for i, (_, row) in enumerate(slopes.iterrows()):
        ax.plot([row["ci low"], row["ci high"]], [i, i], color=COLORS[row["condition"]], lw=2)
        ax.plot(row["slope per year"], i, "o", color=COLORS[row["condition"]], mfc="#fcfcfb", mew=1.5, ms=6)
    ax.axvline(0, color=MUTED, lw=1)
    ax.set_yticks(range(len(slopes))); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlabel("Change in mean span length per year of source age (95% CI)")
    fig.tight_layout()

## 7. Which layer, and how sensitive to the length penalty?
The representation searches store several layers and every search was run at several length penalties. A trend that only appears at one setting is not robust.

In [ ]:
def slope_table(frame, by):
    out = []
    for key, group in frame.groupby(by + ["condition"]):
        group = group.dropna(subset=["mean_span_len", "source_age"])
        if group.story_key.nunique() < 4:
            continue
        out.append({**dict(zip(by + ["condition"], key if isinstance(key, tuple) else (key,))), "slope": age_slope(group)[0]})
    return pd.DataFrame(out)

rep = summaries[summaries.method.ne("natural language") & summaries.length_penalty.eq(penalty)]
by_layer = slope_table(rep, ["method", "layer"])
if len(by_layer):
    fig, axes = plt.subplots(1, by_layer.method.nunique(), figsize=(4.4 * by_layer.method.nunique(), 3.4), sharey=True, squeeze=False)
    for ax, (method, group) in zip(axes[0], by_layer.groupby("method")):
        for condition in CONDITIONS:
            sub = group[group.condition.eq(condition)].sort_values("layer")
            ax.plot(sub.layer, sub.slope, color=COLORS[condition], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=condition)
        ax.axhline(0, color=MUTED, lw=1); ax.set_title(method, loc="left", fontsize=10); ax.set_xlabel("Layer")
    axes[0][0].set_ylabel("Age slope of mean span length"); axes[0][0].legend(frameon=False)
    fig.tight_layout()

In [ ]:
view = summaries[summaries.method.eq("natural language") | summaries.layer.eq(layer)]
by_penalty = slope_table(view, ["method", "length_penalty"])
fig, axes = plt.subplots(1, by_penalty.method.nunique(), figsize=(4.2 * by_penalty.method.nunique(), 3.4), sharey=True, squeeze=False)
for ax, (method, group) in zip(axes[0], by_penalty.groupby("method")):
    for condition in CONDITIONS:
        sub = group[group.condition.eq(condition)].sort_values("length_penalty")
        ax.plot(sub.length_penalty, sub.slope, color=COLORS[condition], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=condition)
    ax.axhline(0, color=MUTED, lw=1); ax.set_title(method, loc="left", fontsize=10); ax.set_xlabel("Length penalty")
axes[0][0].set_ylabel("Age slope of mean span length"); axes[0][0].legend(frameon=False)
fig.tight_layout();

## 8. Do the methods agree?
Story-level mean span length under natural language versus each representation, one point per story and summary.

In [ ]:
from scipy.stats import spearmanr

story = S.pivot_table(index=["story_key", "condition"], columns="method", values="mean_span_len").reset_index()
others = [m for m in SOURCES if m != "natural language" and m in story]
if "natural language" in story and others:
    fig, axes = plt.subplots(1, len(others), figsize=(4.2 * len(others), 3.8), squeeze=False)
    for ax, other in zip(axes[0], others):
        both = story.dropna(subset=["natural language", other])
        for condition in CONDITIONS:
            sub = both[both.condition.eq(condition)]
            ax.scatter(sub["natural language"], sub[other], s=12, alpha=0.6, color=COLORS[condition], label=condition, linewidths=0)
        rho = spearmanr(both["natural language"], both[other]).statistic if len(both) > 2 else float("nan")
        ax.set_title(f"{other}  (Spearman {rho:.2f})", loc="left", fontsize=10)
        ax.set_xlabel("natural language: mean span length"); ax.set_ylabel(f"{other}: mean span length")
    axes[0][0].legend(frameon=False)
    fig.tight_layout()

## Reading these results

- Trust a ratio only where section 4 shows scores above the null.
- Consecutive spans over-widen details that blend distant events, so compare with `best_unconstrained_span_len` in `details.csv`.
- Isolated span vectors are averages of isolated detail vectors, not joint encodings of the span.
- Few stories per age bin means wide intervals; the coverage table is the guide.